# Window Functions Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

%md
## Que1: Ranking Hosts by Total Beds

**Difficulty:** Easy

### Problem

Given an `apartments` table containing host and bed information, calculate the total number of beds managed by each host and assign a dense rank based on the total beds.

Hosts with the same total number of beds receive the same rank, and the next distinct total receives the next consecutive rank.

**Schema columns:** `apartments.apartment_id`, `apartments.host_id`, `apartments.num_beds`, `apartments.city`

**Output columns:** `host_id`, `total_beds`, `host_rank`

Order the result by `host_rank` in ascending order, then by `host_id` in ascending order.

### Examples

#### Example 1

**Input:**

**apartments:**

| apartment_id | host_id | num_beds | city |
|-------------:|--------:|---------:|------|
| 1 | 101 | 2 | SF |
| 2 | 101 | 3 | SF |
| 3 | 102 | 5 | NYC |
| 4 | 103 | 5 | LA |
| 5 | 104 | 1 | SF |

**Output:**

| host_id | total_beds | host_rank |
|--------:|-----------:|----------:|
| 101 | 5 | 1 |
| 102 | 5 | 1 |
| 103 | 5 | 1 |
| 104 | 1 | 2 |

**Explanation:** Host 101 has a total of 5 beds across two apartments. Hosts 102 and 103 also have 5 beds, so all three receive a dense rank of 1. Host 104 has 1 bed and receives the next dense rank, which is 2.

### Constraints

- Use `DENSE_RANK` to assign rankings.
- Hosts with the same total beds receive the same rank.
- Include all hosts.
- Order the result by `host_rank` ascending, then `host_id` ascending.

In [0]:
apartments_data=[(1,101,2,"SF"),(2,101,3,"SF"),(3,102,5,"NYC"),(4,103,5,"LA"),(5,104,1,"SF")]
apartments_df=spark.createDataFrame(apartments_data,["apartment_id","host_id","num_beds","city"])
display(apartments_df)


grouped_df = (
apartments_df.groupBy("host_id").agg(
    sum("num_beds").alias("total_beds")
))

window_spec = Window.orderBy(col("total_beds").desc())

ranked_df = grouped_df.withColumn("host_rank", dense_rank().over(window_spec)).orderBy("host_rank", "host_id")

display(ranked_df)


apartment_id,host_id,num_beds,city
1,101,2,SF
2,101,3,SF
3,102,5,NYC
4,103,5,LA
5,104,1,SF


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


host_id,total_beds,host_rank
101,5,1
102,5,1
103,5,1
104,1,2


%md
## Que2: Third Highest Product Price

**Difficulty:** Easy

### Problem

A merchandising team wants to identify all products that belong to the third-highest distinct price tier.

Products with the same price belong to the same price tier. Return every product whose price is the third-highest distinct price.

**Schema columns:** `products.product_id`, `products.product_name`, `products.price`

**Output columns:** `product_id`, `product_name`, `price`

Order the result by `product_name` in ascending order.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | product_name | price |
|-----------:|--------------|------:|
| 1 | Laptop | 1000 |
| 2 | Monitor | 800 |
| 3 | Desk | 500 |
| 4 | Chair | 500 |
| 5 | Mouse | 50 |

**Output:**

| product_id | product_name | price |
|-----------:|--------------|------:|
| 4 | Chair | 500 |
| 3 | Desk | 500 |

**Explanation:** The distinct product prices are 1000, 800, 500, and 50. The third-highest distinct price is 500, so both products priced at 500 are returned in alphabetical order by product name.

### Constraints

- Consider distinct prices when determining the third-highest price tier.
- Return all products that belong to the third-highest distinct price.
- Order the result by `product_name` in ascending order.

In [0]:
products_data=[(1,"Laptop",1000),(2,"Monitor",800),(3,"Desk",500),(4,"Chair",500),(5,"Mouse",50)]
products_df=spark.createDataFrame(products_data,["product_id","product_name","price"])
display(products_df)

window_spec = Window.orderBy(col("price").desc())

ranked_df = products_df.withColumn("product_rank", dense_rank().over(window_spec))

result_df = ranked_df.filter(col("product_rank") == 3).drop("product_rank").orderBy("product_name")

display(result_df)



product_id,product_name,price
1,Laptop,1000
2,Monitor,800
3,Desk,500
4,Chair,500
5,Mouse,50


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product_id,product_name,price
3,Desk,500
4,Chair,500


%md
## Que3: Dense Rank vs Rank vs Row Number Demo

**Difficulty:** Easy

### Problem

You are given a table of employee salaries. For every employee, calculate three different ranking values based on salary in descending order.

Return the following values for each employee:
- `row_num`: Assign a unique sequential number starting from 1. If multiple employees have the same salary, the employee with the smaller `employee_id` receives the smaller row number.
- `rank_val`: Employees with the same salary receive the same rank, and the next rank skips positions based on the number of tied employees.
- `dense_rank_val`: Employees with the same salary receive the same rank, but the next distinct salary receives the next consecutive rank without any gaps.

**Schema columns:** `employee_salaries.employee_id`, `employee_salaries.name`, `employee_salaries.department`, `employee_salaries.salary`

**Output columns:** `employee_id`, `name`, `department`, `salary`, `row_num`, `rank_val`, `dense_rank_val`

Order the result by `salary` in descending order.

### Examples

#### Example 1

**Input:**

**employee_salaries:**

| employee_id | name | department | salary |
|------------:|------|------------|-------:|
| 1 | Alice | Engineering | 75000 |
| 2 | Bob | Sales | 75000 |
| 3 | Charlie | Engineering | 60000 |
| 4 | Diana | HR | 60000 |
| 6 | Frank | Engineering | 50000 |

**Output:**

| employee_id | name | department | salary | row_num | rank_val | dense_rank_val |
|------------:|------|------------|-------:|--------:|---------:|---------------:|
| 1 | Alice | Engineering | 75000 | 1 | 1 | 1 |
| 2 | Bob | Sales | 75000 | 2 | 1 | 1 |
| 3 | Charlie | Engineering | 60000 | 3 | 3 | 2 |
| 4 | Diana | HR | 60000 | 4 | 3 | 2 |
| 6 | Frank | Engineering | 50000 | 5 | 5 | 3 |

**Explanation:** Alice and Bob share the highest salary, so they receive the same `rank_val` and `dense_rank_val`, while `row_num` remains unique based on the smaller `employee_id`. Charlie and Diana share the next salary and receive the same rankings. Since `RANK` skips values after ties, Charlie receives rank 3, whereas `DENSE_RANK` assigns rank 2 because it does not leave gaps.

### Constraints

- Order the results by salary in descending order.
- Break salary ties for `ROW_NUMBER` using `employee_id` in ascending order.
- `RANK` skips rank values after ties.
- `DENSE_RANK` assigns consecutive ranks without gaps.
- Return all employees.

In [0]:
employee_salaries_data=[(1,"Alice","Engineering",75000),(2,"Bob","Sales",75000),(3,"Charlie","Engineering",60000),(4,"Diana","HR",60000),(6,"Frank","Engineering",50000)]
employee_salaries_df=spark.createDataFrame(employee_salaries_data,["employee_id","name","department","salary"])
display(employee_salaries_df)

window_spec = Window.orderBy(col("salary").desc())

# row_num	rank_val	dense_rank_val
ranked_df = (
employee_salaries_df
    .withColumn("row_num", row_number().over(window_spec))
    .withColumn("rank_val", rank().over(window_spec))
    .withColumn("dense_rank_val", dense_rank().over(window_spec))
)

display(ranked_df)


employee_id,name,department,salary
1,Alice,Engineering,75000
2,Bob,Sales,75000
3,Charlie,Engineering,60000
4,Diana,HR,60000
6,Frank,Engineering,50000


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


employee_id,name,department,salary,row_num,rank_val,dense_rank_val
1,Alice,Engineering,75000,1,1,1
2,Bob,Sales,75000,2,1,1
3,Charlie,Engineering,60000,3,3,2
4,Diana,HR,60000,4,3,2
6,Frank,Engineering,50000,5,5,3


%md
## Que4: Find the Team Size

**Difficulty:** Easy

### Problem

A company organizes employees into teams based on their direct manager. For every employee who has a manager, determine the size of the team reporting to the same manager.

The team size includes all employees sharing the same non-empty `manager_id`, including the employee but excluding the manager.

**Schema columns:** `employees.employee_id`, `employees.manager_id`, `employees.name`

**Output columns:** `employee_id`, `team_size`

Order the result by `employee_id` in ascending order.

### Examples

#### Example 1

**Input:**

**employees:**

| employee_id | manager_id | name |
|------------:|-----------:|------|
| 101 | NULL | CEO |
| 102 | 101 | Alice |
| 103 | 101 | Bob |
| 104 | 102 | Charlie |
| 105 | 102 | Diana |

**Output:**

| employee_id | team_size |
|------------:|----------:|
| 102 | 2 |
| 103 | 2 |
| 104 | 2 |
| 105 | 2 |

**Explanation:** Employees 102 and 103 report to manager 101, so each belongs to a team of size 2. Similarly, employees 104 and 105 report to manager 102, so each also belongs to a team of size 2. Employee 101 is excluded because they do not have a manager.

### Constraints

- Exclude employees whose `manager_id` is `NULL`.
- `team_size` counts all employees sharing the same non-null manager, including the employee and excluding the manager.
- Order the result by `employee_id` in ascending order.

In [0]:
employees_data=[(101,None,"CEO"),(102,101,"Alice"),(103,101,"Bob"),(104,102,"Charlie"),(105,102,"Diana")]
employees_df=spark.createDataFrame(employees_data,["employee_id","manager_id","name"])
display(employees_df)

window_spec = Window.partitionBy("manager_id")

result = (
    employees_df
    .filter(col("manager_id").isNotNull())
    .withColumn("team_size", count("*").over(window_spec))
    .select("employee_id", "team_size")
    .orderBy("employee_id")
)

display(result)


employee_id,manager_id,name
101,null,CEO
102,101,Alice
103,101,Bob
104,102,Charlie
105,102,Diana


employee_id,team_size
102,2
103,2
104,2
105,2


%md
## Que5: Consecutive Available Seats

**Difficulty:** Easy

### Problem

A cinema wants to identify every available seat that has at least one immediately adjacent available seat.

A seat qualifies if it is free and either the previous seat or the next seat is also free.

**Schema columns:** `cinema.seat_id`, `cinema.free`

**Output columns:** `seat_id`

Order the result by `seat_id` in ascending order.

### Examples

#### Example 1

**Input:**

**cinema:**

| seat_id | free |
|--------:|-----:|
| 1 | 1 |
| 2 | 1 |
| 3 | 0 |
| 4 | 1 |
| 5 | 1 |
| 6 | 1 |

**Output:**

| seat_id |
|--------:|
| 1 |
| 2 |
| 4 |
| 5 |
| 6 |

**Explanation:** Seats 1 and 2 are consecutive available seats, so both are included. Seats 4, 5, and 6 form another consecutive block of available seats, so all three are included. Seat 3 is occupied and therefore not returned.

### Constraints

- `free = 1` indicates an available seat and `free = 0` indicates an occupied seat.
- Adjacent seats have `seat_id` values differing by exactly 1.
- Return only available seats that have at least one adjacent available seat.
- Order the result by `seat_id` in ascending order.

In [0]:
cinema_data=[(1,1),(2,1),(3,0),(4,1),(5,1),(6,1)]
cinema_df=spark.createDataFrame(cinema_data,["seat_id","free"])
display(cinema_df)

window_spec = Window.orderBy("seat_id")

prev_next_df = (
cinema_df
    .withColumn("prev_free", lag("free").over(window_spec))
    .withColumn("next_free", lead("free").over(window_spec))
)

free_seat_condition = ((col("free") == 1) & ( (col("prev_free") == 1) | (col("next_free") == 1)  ))

output_df = (
    prev_next_df.filter(free_seat_condition)
    .select("seat_id")
)

display(output_df)

seat_id,free
1,1
2,1
3,0
4,1
5,1
6,1


seat_id
1
2
4
5
6


## Que6: Top 3 Most Profitable Companies

**Difficulty:** Medium

### Problem

Identify the companies that belong to the top three distinct profit levels globally.

Rank companies by profit in descending order using standard competition ranking (`RANK`). Companies with the same profit receive the same rank, and the next rank skips ahead based on the number of tied companies (for example: 1, 1, 3).

Return every company whose profit falls within the top three distinct profit values.

**Schema columns:** `tpc_top_profitable.company_name`, `tpc_top_profitable.profit`

**Output columns:** `rank`, `company_name`, `profit`

Order the result by `rank` in ascending order, then by `company_name` in ascending order.

### Examples

#### Example 1

**Input:**

**tpc_top_profitable:**

| company_name | profit |
|--------------|-------:|
| Amazon | 1000 |
| Apple | 2000 |
| Google | 2000 |
| Microsoft | 1500 |

**Output:**

| rank | company_name | profit |
|-----:|--------------|-------:|
| 1 | Apple | 2000 |
| 1 | Google | 2000 |
| 3 | Microsoft | 1500 |
| 4 | Amazon | 1000 |

**Explanation:** Apple and Google have the highest profit and therefore share rank 1. Since two companies occupy the first rank, Microsoft receives rank 3. Amazon receives rank 4. The top three distinct profit values are 2000, 1500, and 1000, so all four companies are included in the result.

### Constraints

- Use `RANK()` to assign rankings.
- Companies with the same profit receive the same rank.
- Include every company whose profit belongs to the top three distinct profit values.
- Order the result by `rank` ascending, then `company_name` ascending.

In [0]:
tpc_top_profitable_data=[("Amazon",1000),("Apple",2000),("Google",2000),("Microsoft",1500)]
tpc_top_profitable_df=spark.createDataFrame(tpc_top_profitable_data,["company_name","profit"])
display(tpc_top_profitable_df)

window_spec = Window.orderBy(col("profit").desc())

ranked_df = (
tpc_top_profitable_df
    .withColumn("rank", rank().over(window_spec))
    .withColumn("dense_rank", dense_rank().over(window_spec))
    .where("dense_rank <= 3")
    .select("rank", "company_name","profit")
)

display(ranked_df)




company_name,profit
Amazon,1000
Apple,2000
Google,2000
Microsoft,1500


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rank,company_name,profit
1,Apple,2000
1,Google,2000
3,Microsoft,1500
4,Amazon,1000


%md
## Que7: User Third Transaction

**Difficulty:** Medium

### Problem

For each user who has made at least three transactions, find their third transaction in chronological order.

Return the `user_id`, `spend`, and `transaction_date` corresponding to each user's third transaction. Users with fewer than three transactions should not appear in the result.

**Schema columns:** `ttq_transaction_third.user_id`, `ttq_transaction_third.spend`, `ttq_transaction_third.transaction_date`

**Output columns:** `user_id`, `spend`, `transaction_date`

The result may be returned in any order.

### Examples

#### Example 1

**Input:**

**ttq_transaction_third:**

| spend | user_id | transaction_date |
|------:|--------:|------------------|
| 100.5 | 111 | 2022-01-08 |
| 55 | 111 | 2022-01-10 |
| 36 | 121 | 2022-01-18 |
| 24.99 | 145 | 2022-01-26 |
| 89.6 | 111 | 2022-02-05 |

**Output:**

| user_id | spend | transaction_date |
|--------:|------:|------------------|
| 111 | 89.6 | 2022-02-05 |

**Explanation:** User 111 has three transactions ordered by date: January 8, January 10, and February 5. Therefore, the transaction on February 5 with a spend of 89.6 is the third transaction. Users 121 and 145 have fewer than three transactions, so they are excluded.

### Constraints

- Consider transactions in chronological order.
- Transaction dates are unique for each user.
- Return only users with at least three transactions.
- Return the `user_id`, `spend`, and `transaction_date` of each user's third transaction.

In [0]:
ttq_transaction_third_data=[(100.5,111,"2022-01-08"),(55.0,111,"2022-01-10"),(36.0,121,"2022-01-18"),(24.99,145,"2022-01-26"),(89.6,111,"2022-02-05")]
ttq_transaction_third_df=spark.createDataFrame(ttq_transaction_third_data,["spend","user_id","transaction_date"])
display(ttq_transaction_third_df)

window_spec = Window.partitionBy("user_id").orderBy("transaction_date")

ranked_df = (
ttq_transaction_third_df.
    withColumn("rn", row_number().over(window_spec))
)

outpur_df = ranked_df.filter(col("rn") == 3).select("user_id", "spend", "transaction_date")

display(outpur_df)


spend,user_id,transaction_date
100.5,111,2022-01-08
55.0,111,2022-01-10
36.0,121,2022-01-18
24.99,145,2022-01-26
89.6,111,2022-02-05


user_id,spend,transaction_date
111,89.6,2022-02-05


%md
## Que8: Rank Scores Without Gaps

**Difficulty:** Medium

### Problem

A testing service wants to create a leaderboard using only distinct score values.

Return each unique score along with its rank, where the highest distinct score receives rank 1 and each subsequent distinct score receives the next consecutive rank without any gaps.

**Schema columns:** `scores.id`, `scores.score`

**Output columns:** `score`, `rank`

Order the result by `score` in descending order.

### Examples

#### Example 1

**Input:**

**scores:**

| id | score |
|---:|------:|
| 1 | 95 |
| 2 | 95 |
| 3 | 88 |
| 4 | 72 |
| 5 | 72 |

**Output:**

| score | rank |
|------:|-----:|
| 95 | 1 |
| 88 | 2 |
| 72 | 3 |

**Explanation:** The duplicate score of 95 appears only once and receives rank 1. The next distinct scores, 88 and 72, receive consecutive ranks 2 and 3 without any gaps.

### Constraints

- Return one row for each distinct score.
- Equal scores share the same rank.
- Ranks must be consecutive without gaps (`DENSE_RANK` behavior).
- Order the result by `score` in descending order.

In [0]:
scores_data=[(1,95),(2,95),(3,88),(4,72),(5,72)]
scores_df=spark.createDataFrame(scores_data,["id","score"])
display(scores_df)

window_spec = Window.orderBy(col("score").desc())

ranked_df = (
    scores_df.withColumn("rank", dense_rank().over(window_spec))
)

output_df = ranked_df.select("score", "rank").distinct()

display(output_df)


id,score
1,95
2,95
3,88
4,72
5,72


score,rank
95,1
88,2
72,3


%md
## Que9: Report Contiguous Date Ranges

**Difficulty:** Medium

### Problem

Treat consecutive dates with completed tasks as a single project interval. Ignore all tasks that are not completed and treat duplicate completed dates as a single date.

For each contiguous interval of completed dates, return a unique `project_id` along with the interval's `start_date`, `end_date`, and the inclusive number of days in the interval.

**Schema columns:** `tasks.task_id`, `tasks.task_date`, `tasks.status`

**Output columns:** `project_id`, `start_date`, `end_date`, `duration_days`

Order the result by `start_date` in ascending order.

### Examples

#### Example 1

**Input:**

**tasks:**

| task_id | task_date | status |
|--------:|------------|-----------|
| 1 | 2020-01-01 | completed |
| 2 | 2020-01-02 | completed |
| 3 | 2020-01-02 | completed |
| 4 | 2020-01-03 | pending |
| 5 | 2020-01-04 | completed |
| 6 | 2020-01-05 | completed |
| 7 | 2020-01-07 | completed |
| 8 | 2020-01-08 | completed |

**Output:**

| project_id | start_date | end_date | duration_days |
|-----------:|------------|----------|--------------:|
| 1 | 2020-01-01 | 2020-01-02 | 2 |
| 2 | 2020-01-04 | 2020-01-05 | 2 |
| 3 | 2020-01-07 | 2020-01-08 | 2 |

**Explanation:** Duplicate completed tasks on January 2 are treated as a single date. The pending task on January 3 is ignored, breaking the sequence. This results in three contiguous completed-date intervals, each lasting two days.

### Constraints

- Consider only rows where `status = 'completed'`.
- Treat duplicate completed dates as a single date.
- Consecutive calendar dates belong to the same project interval.
- `duration_days` is inclusive of both `start_date` and `end_date`.
- Order the result by `start_date` in ascending order.

In [0]:
tasks_data=[(1,"2020-01-01","completed"),(2,"2020-01-02","completed"),(3,"2020-01-02","completed"),(4,"2020-01-03","pending"),(5,"2020-01-04","completed"),(6,"2020-01-05","completed"),(7,"2020-01-07","completed"),(8,"2020-01-08","completed")]
tasks_df=spark.createDataFrame(tasks_data,["task_id","task_date","status"])
display(tasks_df)


completed_dates_df = tasks_df.filter(col("status") == "completed").select(col("task_date").cast("date")).distinct()

window_spec = Window.orderBy(col("task_date"))

new_project_condition = ((col("prev_date").isNull()) | (col("task_date") - 1 != col("prev_date")))

prev_dates_df = (
completed_dates_df
    .withColumn("prev_date", lag(col("task_date")).over(window_spec))
    .withColumn("is_new_project", when(new_project_condition, 1).otherwise(0))
)

project_id_df = (
prev_dates_df
    .withColumn("project_id", sum("is_new_project").over(window_spec))
)

groped_df = (
project_id_df.groupBy("project_id").agg(
    min("task_date").alias("start_date"),
    max("task_date").alias("end_date")
)
.withColumn("duration", datediff(col("end_date"), col("start_date")) + 1)
.orderBy("start_date")
)


display(groped_df)



task_id,task_date,status
1,2020-01-01,completed
2,2020-01-02,completed
3,2020-01-02,completed
4,2020-01-03,pending
5,2020-01-04,completed
6,2020-01-05,completed
7,2020-01-07,completed
8,2020-01-08,completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


project_id,start_date,end_date,duration
1,2020-01-01,2020-01-02,2
2,2020-01-04,2020-01-05,2
3,2020-01-07,2020-01-08,2


%md
## Que10: First Successful Charge in January 2024

**Difficulty:** Medium

### Problem

A payments team wants to identify customers whose first-ever successful charge occurred in January 2024.

Consider only charges with a status of `succeeded` when determining a customer's first successful charge. Earlier failed or pending charges do not affect the result.

Return each qualifying customer's ID, the date of their first successful charge, and the corresponding charge amount.

**Schema columns:** `charges.charge_id`, `charges.customer_id`, `charges.status`, `charges.amount`, `charges.charged_at`

**Output columns:** `customer_id`, `first_charge_date`, `amount`

Order the result by `customer_id` in ascending order.

### Examples

#### Example 1

**Input:**

**charges:**

| charge_id | customer_id | status | amount | charged_at |
|----------:|------------:|----------|-------:|------------|
| 1 | 1001 | failed | 50 | 2023-11-15 |
| 2 | 1001 | succeeded | 75 | 2023-12-20 |
| 3 | 1001 | succeeded | 100 | 2024-01-15 |
| 4 | 1002 | failed | 30 | 2023-12-01 |
| 5 | 1002 | pending | 45 | 2023-12-15 |
| 6 | 1002 | succeeded | 60 | 2024-01-10 |

**Output:**

| customer_id | first_charge_date | amount |
|------------:|-------------------|-------:|
| 1002 | 2024-01-10 | 60 |

**Explanation:** Customer 1001 had a successful charge before January 2024, so they are excluded. Customer 1002's first successful charge occurred on January 10, 2024, making them eligible for the result.

### Constraints

- Consider only rows where `status = 'succeeded'`.
- A customer's first successful charge must occur between `2024-01-01` (inclusive) and `2024-02-01` (exclusive).
- Earlier failed or pending charges do not affect eligibility.
- Order the result by `customer_id` in ascending order.

In [0]:
charges_data=[(1,1001,"failed",50,"2023-11-15"),(2,1001,"succeeded",75,"2023-12-20"),(3,1001,"succeeded",100,"2024-01-15"),(4,1002,"failed",30,"2023-12-01"),(5,1002,"pending",45,"2023-12-15"),(6,1002,"succeeded",60,"2024-01-10")]
charges_df=spark.createDataFrame(charges_data,["charge_id","customer_id","status","amount","charged_at"])
display(charges_df)

filtered_charges_df = (
    charges_df
    .withColumn("charged_at", col("charged_at").cast("date"))
    .where(
        (col("charged_at").between("2024-01-01", "2024-01-31")) &
        (col("status") == "succeeded")
    )
)


window_spec = Window.orderBy("charged_at")

ranked_df = (
filtered_charges_df.withColumn("rn", row_number().over(window_spec))
.where(col("rn") == 1)
.select("customer_id", col("charged_at").alias("first_charge_date"), col("amount"))
)


display(ranked_df)


charge_id,customer_id,status,amount,charged_at
1,1001,failed,50,2023-11-15
2,1001,succeeded,75,2023-12-20
3,1001,succeeded,100,2024-01-15
4,1002,failed,30,2023-12-01
5,1002,pending,45,2023-12-15
6,1002,succeeded,60,2024-01-10


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,first_charge_date,amount
1002,2024-01-10,60


%md
## Que11: Stock Price Analysis with Lag and Lead

**Difficulty:** Medium

### Problem

A market analyst wants to analyze each stock's daily closing prices by comparing them with the immediately previous and next recorded prices for the same ticker.

For every stock price record, return the previous closing price, the current closing price, the next closing price, and the percentage change from the previous recorded closing price to the current closing price. Round the percentage change to two decimal places.

**Schema columns:** `stock_prices.price_id`, `stock_prices.ticker`, `stock_prices.price_date`, `stock_prices.closing_price`

**Output columns:** `ticker`, `price_date`, `prev_price`, `closing_price`, `next_price`, `daily_change_pct`

Order the result by `ticker` in ascending order, then by `price_date` in ascending order.

### Examples

#### Example 1

**Input:**

**stock_prices:**

| price_id | ticker | price_date | closing_price |
|---------:|--------|------------|--------------:|
| 1 | AAPL | 2026-01-05 | 150.25 |
| 2 | AAPL | 2026-01-06 | 152.10 |
| 3 | AAPL | 2026-01-07 | 151.50 |

**Output:**

| ticker | price_date | prev_price | closing_price | next_price | daily_change_pct |
|--------|------------|-----------:|--------------:|-----------:|-----------------:|
| AAPL | 2026-01-05 | NULL | 150.25 | 152.10 | NULL |
| AAPL | 2026-01-06 | 150.25 | 152.10 | 151.50 | 1.23 |
| AAPL | 2026-01-07 | 152.10 | 151.50 | NULL | -0.39 |

**Explanation:** The first record has no previous closing price, so both `prev_price` and `daily_change_pct` are `NULL`. The last record has no next closing price, so `next_price` is `NULL`. All comparisons are made within the same ticker based on the chronological order of recorded prices.

### Constraints

- Previous and next prices are determined within the same ticker ordered by `price_date`.
- Leave unavailable previous, next, or percentage values as `NULL`.
- Round `daily_change_pct` to two decimal places.
- Order the result by `ticker` ascending, then `price_date` ascending.

In [0]:
stock_prices_data=[(1,"AAPL","2026-01-05",150.25),(2,"AAPL","2026-01-06",152.10),(3,"AAPL","2026-01-07",151.50)]
stock_prices_df=spark.createDataFrame(stock_prices_data,["price_id","ticker","price_date","closing_price"])
display(stock_prices_df)



window_spec = Window.partitionBy("ticker").orderBy("price_date")

# Previous Price

complete_df = (
stock_prices_df.
    withColumn("prev_price", lag("closing_price").over(window_spec)).
    withColumn("next_price", lead("closing_price").over(window_spec)).
	withColumn("daily_change_pct", round(((col("closing_price") - col("prev_price")) / col("prev_price")) * 100, 2))
    # select # ticker	price_date	prev_price	closing_price	next_price	daily_change_pct
    .select("ticker", "price_date", "prev_price", "closing_price", "next_price", "daily_change_pct")
    .orderBy("ticker", "price_date")
)

display(complete_df)


price_id,ticker,price_date,closing_price
1,AAPL,2026-01-05,150.25
2,AAPL,2026-01-06,152.1
3,AAPL,2026-01-07,151.5


ticker,price_date,prev_price,closing_price,next_price,daily_change_pct
AAPL,2026-01-05,null,150.25,152.1,null
AAPL,2026-01-06,150.25,152.1,151.5,1.23
AAPL,2026-01-07,152.1,151.5,null,-0.39


## Que12: Zillow Home Price Moving Average by ZIP

**Difficulty:** Medium

### Problem

Zillow wants to smooth fluctuations in home sale prices by calculating a rolling 30-day average for each ZIP code.

For every sale date within a ZIP code, consider all sales from the same ZIP whose `sale_date` falls within the 30-day calendar window ending on that date (the current date plus the previous 29 days). Return the moving average price (rounded to 2 decimal places), the minimum price, and the maximum price for that window.

**Schema columns:** `home_prices.listing_id`, `home_prices.zip_code`, `home_prices.price`, `home_prices.sale_date`

**Output columns:** `zip_code`, `update_date`, `moving_avg_price`, `min_price`, `max_price`

Order the result by `zip_code` in ascending order, then by `update_date` in ascending order.

### Examples

#### Example 1

**Input:**

**home_prices:**

| listing_id | zip_code | price | sale_date |
|-----------:|---------:|------:|------------|
| 4 | 10001 | 530000 | 2026-02-15 |
| 5 | 10001 | 540000 | 2026-03-05 |
| 6 | 10001 | 545000 | 2026-03-15 |
| 8 | 10002 | 450000 | 2026-01-10 |
| 9 | 10002 | 460000 | 2026-01-20 |
| 11 | 10002 | 480000 | 2026-02-20 |

**Output:**

| zip_code | update_date | moving_avg_price | min_price | max_price |
|---------:|-------------|-----------------:|----------:|----------:|
| 10001 | 2026-02-15 | 530000.00 | 530000 | 530000 |
| 10001 | 2026-03-05 | 535000.00 | 530000 | 540000 |
| 10001 | 2026-03-15 | 538333.33 | 530000 | 545000 |
| 10002 | 2026-01-10 | 450000.00 | 450000 | 450000 |
| 10002 | 2026-01-20 | 455000.00 | 450000 | 460000 |
| 10002 | 2026-02-20 | 480000.00 | 480000 | 480000 |

**Explanation:** For each sale date, only sales from the same ZIP code that occurred within the current day and the previous 29 calendar days are included in the calculation. The moving average is rounded to two decimal places, while the minimum and maximum prices are reported exactly.

### Constraints

- Use a 30-day calendar window consisting of the current `sale_date` and the previous 29 days.
- Calculate values separately for each `zip_code`.
- Round `moving_avg_price` to 2 decimal places.
- Return one row for each distinct `(zip_code, update_date)` pair.
- Order the result by `zip_code` ascending, then `update_date` ascending.

In [0]:
home_prices_data=[(4,10001,530000,"2026-02-15"),(5,10001,540000,"2026-03-05"),(6,10001,545000,"2026-03-15"),(8,10002,450000,"2026-01-10"),(9,10002,460000,"2026-01-20"),(11,10002,480000,"2026-02-20")]
home_prices_df=spark.createDataFrame(home_prices_data,["listing_id","zip_code","price","sale_date"])
display(home_prices_df)

window_spec = Window.partitionBy("zip_code").orderBy("sale_date").rowsBetween(-29, 0)

output_df = (
home_prices_df
    .withColumn("moving_avg_price", round((avg("price").over(window_spec)), 2))
    .withColumn("min_price", min("price").over(window_spec))
    .withColumn("max_price", max("price").over(window_spec))
    .withColumnRenamed("sale_date", "update_date")
    .select("zip_code", "update_date", "moving_avg_price", "min_price", "max_price")
    .orderBy("zip_code", "update_date")
)

display(output_df)




listing_id,zip_code,price,sale_date
4,10001,530000,2026-02-15
5,10001,540000,2026-03-05
6,10001,545000,2026-03-15
8,10002,450000,2026-01-10
9,10002,460000,2026-01-20
11,10002,480000,2026-02-20


zip_code,update_date,moving_avg_price,min_price,max_price
10001,2026-02-15,530000.0,530000,530000
10001,2026-03-05,535000.0,530000,540000
10001,2026-03-15,538333.33,530000,545000
10002,2026-01-10,450000.0,450000,450000
10002,2026-01-20,455000.0,450000,460000
10002,2026-02-20,463333.33,450000,480000


%md
## Que13: Products with 50% Sales Increase

**Difficulty:** Medium

### Problem

Amazon product managers want to identify products that experienced significant month-over-month sales growth.

For each product, compare its sales in a month with its sales in the immediately preceding recorded month. Return only those months where the sales increased by at least 50%.

For every qualifying month, report the product ID, the previous month's sales, the current month's sales, and the percentage increase rounded to two decimal places.

**Schema columns:** `monthly_sales.product_id`, `monthly_sales.month`, `monthly_sales.sales_amount`

**Output columns:** `product_id`, `previous_month_sales`, `sales_amount`, `percent_increase`

Order the result by `percent_increase` in descending order. If multiple rows have the same percentage increase, order by `product_id` in ascending order.

### Examples

#### Example 1

**Input:**

**monthly_sales:**

| product_id | month | sales_amount |
|------------|------------|-------------:|
| P001 | 2024-01-01 | 1000 |
| P001 | 2024-02-01 | 1200 |
| P001 | 2024-03-01 | 1800 |
| P004 | 2024-01-01 | 1500 |
| P004 | 2024-02-01 | 2000 |
| P004 | 2024-03-01 | 3200 |
| P007 | 2024-02-01 | 500 |
| P007 | 2024-03-01 | 700 |

**Output:**

| product_id | previous_month_sales | sales_amount | percent_increase |
|------------|---------------------:|-------------:|-----------------:|
| P004 | 2000 | 3200 | 60.00 |
| P001 | 1200 | 1800 | 50.00 |

**Explanation:** Product P004 increased from 2000 to 3200, resulting in a 60.00% increase. Product P001 increased from 1200 to 1800, which is exactly a 50.00% increase and therefore qualifies. Product P007 increased by only 40.00%, so it is excluded. The first recorded month for each product is ignored because there is no previous month for comparison.

### Constraints

- Compare each month with the immediately preceding recorded month for the same product.
- Exclude the first recorded month for every product.
- Calculate `percent_increase` as `((current - previous) / previous) * 100`, rounded to 2 decimal places.
- Include only rows where `percent_increase` is at least 50%.
- Order the result by `percent_increase` descending, then `product_id` ascending.

In [0]:
monthly_sales_data=[("P001","2024-01-01",1000),("P001","2024-02-01",1200),("P001","2024-03-01",1800),("P004","2024-01-01",1500),("P004","2024-02-01",2000),("P004","2024-03-01",3200),("P007","2024-02-01",500),("P007","2024-03-01",700)]
monthly_sales_df=spark.createDataFrame(monthly_sales_data,["product_id","month","sales_amount"])
display(monthly_sales_df)

window_spec = Window.partitionBy("product_id").orderBy("month")

# product_id	previous_month_sales	sales_amount	percent_increase

prev_month_sales_df = (
monthly_sales_df
    .withColumn("previous_month_sales", lag("sales_amount").over(window_spec))
    .withColumn("percent_increase", round( ((col("sales_amount") - col("previous_month_sales")) / col("previous_month_sales")) * 100    , 2))
)

output_df = (
prev_month_sales_df
    .where("percent_increase >= 50")
    .select("product_id", "previous_month_sales", "sales_amount", "percent_increase")
    .orderBy(col("percent_increase").desc(), col("product_id"))
)

display(output_df)



product_id,month,sales_amount
P001,2024-01-01,1000
P001,2024-02-01,1200
P001,2024-03-01,1800
P004,2024-01-01,1500
P004,2024-02-01,2000
P004,2024-03-01,3200
P007,2024-02-01,500
P007,2024-03-01,700


product_id,previous_month_sales,sales_amount,percent_increase
P004,2000,3200,60.0
P001,1200,1800,50.0


%md
## Que14: Lead and Lag Functions

**Difficulty:** Medium

### Problem

A trading analytics platform wants to display each stock's closing price alongside the previous and next trading day's closing prices for the same stock. It also needs to calculate the percentage change from the previous trading day's closing price.

For every record in the `stock_prices` table, return the previous closing price, the next closing price, and the daily percentage change. If there is no previous trading day, return `0` for `prev_price` and `NULL` for `daily_change_pct`. If there is no next trading day, return `0` for `next_price`.

**Schema columns:** `stock_prices.trade_date`, `stock_prices.stock_symbol`, `stock_prices.closing_price`

**Output columns:** `trade_date`, `stock_symbol`, `closing_price`, `prev_price`, `next_price`, `daily_change_pct`

Order the result by `stock_symbol` in ascending order, then by `trade_date` in ascending order.

### Examples

#### Example 1

**Input:**

**stock_prices:**

| trade_date | stock_symbol | closing_price |
|------------|--------------|--------------:|
| 2024-01-01 | AAPL | 95.17 |
| 2024-01-02 | AAPL | 104.42 |
| 2024-01-03 | AAPL | 98.85 |
| 2024-01-04 | AAPL | 99.49 |
| 2024-01-01 | GOOGL | 96.32 |
| 2024-01-02 | GOOGL | 95.87 |
| 2024-01-03 | GOOGL | 96.82 |

**Output:**

| trade_date | stock_symbol | closing_price | prev_price | next_price | daily_change_pct |
|------------|--------------|--------------:|-----------:|-----------:|-----------------:|
| 2024-01-01 | AAPL | 95.17 | 0 | 104.42 | NULL |
| 2024-01-02 | AAPL | 104.42 | 95.17 | 98.85 | 9.72 |
| 2024-01-03 | AAPL | 98.85 | 104.42 | 99.49 | -5.33 |
| 2024-01-04 | AAPL | 99.49 | 98.85 | 0 | 0.65 |
| 2024-01-01 | GOOGL | 96.32 | 0 | 95.87 | NULL |
| 2024-01-02 | GOOGL | 95.87 | 96.32 | 96.82 | -0.47 |
| 2024-01-03 | GOOGL | 96.82 | 95.87 | 0 | 0.99 |

**Explanation:** Previous and next prices are determined within each stock symbol based on trading date. The first trading day for each stock has no previous close, so `prev_price` is `0` and `daily_change_pct` is `NULL`. The last trading day has no next close, so `next_price` is `0`.

### Constraints

- Calculate previous and next prices within each `stock_symbol` ordered by `trade_date`.
- Use `0` for missing previous or next prices.
- Calculate `daily_change_pct` as `((closing_price - prev_price) / prev_price) * 100`, rounded to 2 decimal places.
- Set `daily_change_pct` to `NULL` when there is no previous trading day.
- Order the result by `stock_symbol` ascending, then `trade_date` ascending.

In [0]:
stock_prices_data=[("2024-01-01","AAPL",95.17),("2024-01-02","AAPL",104.42),("2024-01-03","AAPL",98.85),("2024-01-04","AAPL",99.49),("2024-01-01","GOOGL",96.32),("2024-01-02","GOOGL",95.87),("2024-01-03","GOOGL",96.82)]
stock_prices_df=spark.createDataFrame(stock_prices_data,["trade_date","stock_symbol","closing_price"])
display(stock_prices_df)

window_spec = Window.partitionBy("stock_symbol").orderBy("trade_date")

# Previous Price

complete_df = (
stock_prices_df.
    withColumn("prev_price", lag("closing_price").over(window_spec)).
    withColumn("next_price", lead("closing_price").over(window_spec)).
	withColumn("daily_change_pct", round(((col("closing_price") - col("prev_price")) / col("prev_price")) * 100, 2))
    # select # stock_symbol	trade_date	prev_price	closing_price	next_price	daily_change_pct
    .select("stock_symbol", "trade_date", "prev_price", "closing_price", "next_price", "daily_change_pct")
    .fillna(0, subset=["prev_price", "next_price"])
    .orderBy("stock_symbol", "trade_date")
)

display(complete_df)



trade_date,stock_symbol,closing_price
2024-01-01,AAPL,95.17
2024-01-02,AAPL,104.42
2024-01-03,AAPL,98.85
2024-01-04,AAPL,99.49
2024-01-01,GOOGL,96.32
2024-01-02,GOOGL,95.87
2024-01-03,GOOGL,96.82


stock_symbol,trade_date,prev_price,closing_price,next_price,daily_change_pct
AAPL,2024-01-01,0.0,95.17,104.42,null
AAPL,2024-01-02,95.17,104.42,98.85,9.72
AAPL,2024-01-03,104.42,98.85,99.49,-5.33
AAPL,2024-01-04,98.85,99.49,0.0,0.65
GOOGL,2024-01-01,0.0,96.32,95.87,null
GOOGL,2024-01-02,96.32,95.87,96.82,-0.47
GOOGL,2024-01-03,95.87,96.82,0.0,0.99


%md
## Que15: Rank and Dense Rank by Group

**Difficulty:** Medium

### Problem

A retailer wants to compare two ranking methods for sales within each store. For every sale, calculate both the standard rank and the dense rank based on the sale amount within the same store.

Higher sale amounts receive better ranks. When multiple sales have the same amount, they share the same rank. In `sale_rank`, ties leave gaps in subsequent rankings, while in `sale_dense_rank`, ties do not leave gaps.

**Schema columns:** `store_sales.sale_id`, `store_sales.store_id`, `store_sales.sale_date`, `store_sales.amount`

**Output columns:** `store_id`, `sale_date`, `amount`, `sale_rank`, `sale_dense_rank`

Order the result by `store_id` in ascending order, `amount` in descending order, and `sale_date` in ascending order.

### Examples

#### Example 1

**Input:**

**store_sales:**

| sale_id | store_id | sale_date | amount |
|--------:|----------|------------|-------:|
| 1 | A | 2024-01-01 | 100 |
| 2 | A | 2024-01-02 | 100 |
| 3 | A | 2024-01-03 | 80 |
| 4 | A | 2024-01-04 | 60 |
| 5 | B | 2024-01-01 | 50 |

**Output:**

| store_id | sale_date | amount | sale_rank | sale_dense_rank |
|----------|------------|-------:|----------:|----------------:|
| A | 2024-01-01 | 100 | 1 | 1 |
| A | 2024-01-02 | 100 | 1 | 1 |
| A | 2024-01-03 | 80 | 3 | 2 |
| A | 2024-01-04 | 60 | 4 | 3 |
| B | 2024-01-01 | 50 | 1 | 1 |

**Explanation:** Within Store A, the two sales of 100 share rank 1. The sale of 80 receives rank 3 because standard ranking skips a position after ties, while its dense rank is 2 because dense ranking counts only distinct sale amounts. Store B contains only one sale, so both ranks are 1.

### Constraints

- Calculate rankings independently for each `store_id`.
- Rank higher `amount` values before lower values.
- Equal amounts receive the same rank in both ranking methods.
- `sale_rank` leaves gaps after ties, while `sale_dense_rank` does not.
- Order the result by `store_id` ascending, `amount` descending, then `sale_date` ascending.

In [0]:
store_sales_data=[(1,"A","2024-01-01",100),(2,"A","2024-01-02",100),(3,"A","2024-01-03",80),(4,"A","2024-01-04",60),(5,"B","2024-01-01",50)]
store_sales_df=spark.createDataFrame(store_sales_data,["sale_id","store_id","sale_date","amount"])
display(store_sales_df)

window_spec = Window.partitionBy("store_id").orderBy(col("amount").desc())

output_df = (
store_sales_df
    .withColumn("sale_rank", rank().over(window_spec))
    .withColumn("sale_dense_rank", dense_rank().over(window_spec))
    # store_id	sale_date	amount	sale_rank	sale_dense_rank
    .select("store_id", "sale_date", "amount", "sale_rank", "sale_dense_rank")
    .orderBy("store_id", col("amount").desc(), "sale_date")
)

display(output_df)


sale_id,store_id,sale_date,amount
1,A,2024-01-01,100
2,A,2024-01-02,100
3,A,2024-01-03,80
4,A,2024-01-04,60
5,B,2024-01-01,50


store_id,sale_date,amount,sale_rank,sale_dense_rank
A,2024-01-01,100,1,1
A,2024-01-02,100,1,1
A,2024-01-03,80,3,2
A,2024-01-04,60,4,3
B,2024-01-01,50,1,1


%md
## Que16: Percentage Change and Growth Rates

**Difficulty:** Medium

### Problem

A product team wants to analyze month-over-month revenue growth for each product.

For every monthly revenue record, return the previous month's revenue for the same product and calculate the percentage change from the previous month. The percentage change is calculated as:

`((revenue - prev_revenue) / prev_revenue) * 100`

Round the percentage change to two decimal places. For the first recorded month of each product, both `prev_revenue` and `pct_change` should be `NULL`.

**Schema columns:** `monthly_revenue.month`, `monthly_revenue.product`, `monthly_revenue.revenue`

**Output columns:** `month`, `product`, `revenue`, `prev_revenue`, `pct_change`

Order the result by `product` in ascending order, then by `month` in ascending order.

### Examples

#### Example 1

**Input:**

**monthly_revenue:**

| month | product | revenue |
|--------|----------|--------:|
| 2024-01 | Product A | 1000.00 |
| 2024-02 | Product A | 1100.00 |
| 2024-03 | Product A | 1050.00 |
| 2024-04 | Product A | 1200.00 |
| 2024-05 | Product A | 1150.00 |

**Output:**

| month | product | revenue | prev_revenue | pct_change |
|--------|----------|--------:|-------------:|-----------:|
| 2024-01 | Product A | 1000.00 | NULL | NULL |
| 2024-02 | Product A | 1100.00 | 1000.00 | 10.00 |
| 2024-03 | Product A | 1050.00 | 1100.00 | -4.55 |
| 2024-04 | Product A | 1200.00 | 1050.00 | 14.29 |
| 2024-05 | Product A | 1150.00 | 1200.00 | -4.17 |

**Explanation:** Each month's revenue is compared with the immediately previous month's revenue for the same product. The first month has no previous revenue, so both `prev_revenue` and `pct_change` are `NULL`.

### Constraints

- Calculate values independently for each `product`.
- `prev_revenue` is the revenue from the immediately previous month.
- The first month for each product must have `NULL` for both `prev_revenue` and `pct_change`.
- Round `pct_change` to 2 decimal places.
- Order the result by `product` ascending, then `month` ascending.

In [0]:
monthly_revenue_data=[("2024-01","Product A",1000.00),("2024-02","Product A",1100.00),("2024-03","Product A",1050.00),("2024-04","Product A",1200.00),("2024-05","Product A",1150.00)]
monthly_revenue_df=spark.createDataFrame(monthly_revenue_data,["month","product","revenue"])
display(monthly_revenue_df)

window_spec = Window.partitionBy("product").orderBy(col("month"))

output_df = (
monthly_revenue_df
    .withColumn("prev_month_revenue", lag("revenue").over(window_spec))
    .withColumn("pct_change", round(((col("revenue") - col("prev_month_revenue")) / col("prev_month_revenue")) * 100, 2))
    .orderBy("product", "month")
)

display(output_df)



month,product,revenue
2024-01,Product A,1000.0
2024-02,Product A,1100.0
2024-03,Product A,1050.0
2024-04,Product A,1200.0
2024-05,Product A,1150.0


month,product,revenue,prev_month_revenue,pct_change
2024-01,Product A,1000.0,null,null
2024-02,Product A,1100.0,1000.0,10.0
2024-03,Product A,1050.0,1100.0,-4.55
2024-04,Product A,1200.0,1050.0,14.29
2024-05,Product A,1150.0,1200.0,-4.17


%md
## Que17: Rolling Window Calculations

**Difficulty:** Medium

### Problem

A market-data team wants to smooth daily stock price movements by calculating rolling statistics for each stock ticker.

For every trading date, calculate the rolling average, maximum, and minimum closing price using the current trading row and the two previous trading rows for the same ticker. If fewer than three rows are available, use all available rows up to the current trading date.

Round the rolling average to two decimal places.

**Schema columns:** `daily_stock.trade_date`, `daily_stock.ticker`, `daily_stock.close_price`, `daily_stock.volume`

**Output columns:** `trade_date`, `ticker`, `close_price`, `rolling_3day_avg`, `rolling_3day_max`, `rolling_3day_min`

Order the result by `ticker` in ascending order, then by `trade_date` in ascending order.

### Examples

#### Example 1

**Input:**

**daily_stock:**

| trade_date | ticker | close_price | volume |
|------------|--------|------------:|-------:|
| 2024-01-01 | AAPL | 106.71 | 3361366 |
| 2024-01-02 | AAPL | 126.55 | 1307277 |
| 2024-01-03 | AAPL | 128.06 | 3121834 |
| 2024-01-04 | AAPL | 118.67 | 4914373 |
| 2024-01-05 | AAPL | 112.63 | 1288531 |

**Output:**

| trade_date | ticker | close_price | rolling_3day_avg | rolling_3day_max | rolling_3day_min |
|------------|--------|------------:|-----------------:|-----------------:|-----------------:|
| 2024-01-01 | AAPL | 106.71 | 106.71 | 106.71 | 106.71 |
| 2024-01-02 | AAPL | 126.55 | 116.63 | 126.55 | 106.71 |
| 2024-01-03 | AAPL | 128.06 | 120.44 | 128.06 | 106.71 |
| 2024-01-04 | AAPL | 118.67 | 124.43 | 128.06 | 118.67 |
| 2024-01-05 | AAPL | 112.63 | 119.79 | 128.06 | 112.63 |

**Explanation:** For each trading day, the rolling calculations use the current row and up to the previous two trading rows for the same ticker. For example, on 2024-01-04, the window contains closing prices 126.55, 128.06, and 118.67, producing an average of 124.43, a maximum of 128.06, and a minimum of 118.67.

### Constraints

- Perform calculations independently for each `ticker`.
- The rolling window includes the current row and the previous two trading rows.
- Use all available rows for the first and second observations of each ticker.
- Round `rolling_3day_avg` to 2 decimal places.
- Order the result by `ticker` ascending, then `trade_date` ascending.

In [0]:
daily_stock_data=[("2024-01-01","AAPL",106.71,3361366),("2024-01-02","AAPL",126.55,1307277),("2024-01-03","AAPL",128.06,3121834),("2024-01-04","AAPL",118.67,4914373),("2024-01-05","AAPL",112.63,1288531)]
daily_stock_df=spark.createDataFrame(daily_stock_data,["trade_date","ticker","close_price","volume"])
display(daily_stock_df)

window_spec = Window.partitionBy("ticker").orderBy("trade_date").rowsBetween(-2, 0)

# rolling_3day_avg	rolling_3day_max	rolling_3day_min
# select trade_date	ticker	close_price	rolling_3day_avg	rolling_3day_max	rolling_3day_min
output_df = (
daily_stock_df
    .withColumn("rolling_3day_avg", round(avg("close_price").over(window_spec), 2))
    .withColumn("rolling_3day_max", max("close_price").over(window_spec))
    .withColumn("rolling_3day_min", min("close_price").over(window_spec))
    .select("trade_date", "ticker", "close_price", "rolling_3day_avg", "rolling_3day_max", "rolling_3day_min")
    .orderBy("ticker", "trade_date")
)

display(output_df)




trade_date,ticker,close_price,volume
2024-01-01,AAPL,106.71,3361366
2024-01-02,AAPL,126.55,1307277
2024-01-03,AAPL,128.06,3121834
2024-01-04,AAPL,118.67,4914373
2024-01-05,AAPL,112.63,1288531


trade_date,ticker,close_price,rolling_3day_avg,rolling_3day_max,rolling_3day_min
2024-01-01,AAPL,106.71,106.71,106.71,106.71
2024-01-02,AAPL,126.55,116.63,126.55,106.71
2024-01-03,AAPL,128.06,120.44,128.06,106.71
2024-01-04,AAPL,118.67,124.43,128.06,118.67
2024-01-05,AAPL,112.63,119.79,128.06,112.63


%md
## Que18: Duplicate Records Detection

**Difficulty:** Medium

### Problem

A CRM team wants to identify duplicate contacts based on their first name, last name, and email address.

Group contacts that share the same `(first_name, last_name, email)` combination. Assign each distinct group a unique group ID starting from 1 in alphabetical order of first name, last name, and email. Within each group, mark the most recently created contact as the primary contact.

**Schema columns:** `contacts.contact_id`, `contacts.first_name`, `contacts.last_name`, `contacts.email`, `contacts.phone`, `contacts.created_at`

**Output columns:** `contact_id`, `first_name`, `last_name`, `email`, `dup_group_id`, `is_primary`

Order the result by `first_name`, `last_name`, `email`, and `contact_id` in ascending order.

### Examples

#### Example 1

**Input:**

**contacts:**

| contact_id | first_name | last_name | email | phone | created_at |
|-----------:|------------|-----------|-------|---------|------------|
| 1 | John | Doe | john.doe@example.com | 555-0001 | 2024-01-01 |
| 2 | John | Doe | john.doe@example.com | 555-0002 | 2024-01-05 |
| 3 | Jane | Smith | jane.smith@example.com | 555-0003 | 2024-01-02 |
| 4 | Jane | Smith | jane.smith@example.com | 555-0004 | 2024-01-10 |
| 5 | Bob | Johnson | bob.johnson@example.com | 555-0005 | 2024-01-03 |
| 6 | Alice | Williams | alice.williams@example.com | 555-0006 | 2024-01-04 |
| 7 | Alice | Williams | alice.williams@example.com | 555-0007 | 2024-01-08 |
| 8 | Charlie | Brown | charlie.brown@example.com | 555-0008 | 2024-01-06 |
| 9 | Eve | Davis | eve.davis@example.com | 555-0009 | 2024-01-07 |
| 10 | Eve | Davis | eve.davis@example.com | 555-0010 | 2024-01-12 |

**Output:**

| contact_id | first_name | last_name | email | dup_group_id | is_primary |
|-----------:|------------|-----------|-------|-------------:|-----------:|
| 6 | Alice | Williams | alice.williams@example.com | 1 | 0 |
| 7 | Alice | Williams | alice.williams@example.com | 1 | 1 |
| 5 | Bob | Johnson | bob.johnson@example.com | 2 | 1 |
| 8 | Charlie | Brown | charlie.brown@example.com | 3 | 1 |
| 9 | Eve | Davis | eve.davis@example.com | 4 | 0 |
| 10 | Eve | Davis | eve.davis@example.com | 4 | 1 |
| 3 | Jane | Smith | jane.smith@example.com | 5 | 0 |
| 4 | Jane | Smith | jane.smith@example.com | 5 | 1 |
| 1 | John | Doe | john.doe@example.com | 6 | 0 |
| 2 | John | Doe | john.doe@example.com | 6 | 1 |

**Explanation:** Contacts are grouped by first name, last name, and email. Each unique group receives a sequential ID in alphabetical order. Within every group, the contact with the latest `created_at` value is marked as the primary contact (`is_primary = 1`), while all others are marked as `0`.

### Constraints

- Group contacts by `(first_name, last_name, email)`.
- Assign `dup_group_id` starting from 1 in alphabetical order of the grouping columns.
- Set `is_primary` to `1` for the contact with the latest `created_at` in each group; otherwise `0`.
- Order the result by `first_name`, `last_name`, `email`, and `contact_id` in ascending order.

In [0]:
contacts_data=[(1,"John","Doe","john.doe@example.com","555-0001","2024-01-01"),(2,"John","Doe","john.doe@example.com","555-0002","2024-01-05"),(3,"Jane","Smith","jane.smith@example.com","555-0003","2024-01-02"),(4,"Jane","Smith","jane.smith@example.com","555-0004","2024-01-10"),(5,"Bob","Johnson","bob.johnson@example.com","555-0005","2024-01-03"),(6,"Alice","Williams","alice.williams@example.com","555-0006","2024-01-04"),(7,"Alice","Williams","alice.williams@example.com","555-0007","2024-01-08"),(8,"Charlie","Brown","charlie.brown@example.com","555-0008","2024-01-06"),(9,"Eve","Davis","eve.davis@example.com","555-0009","2024-01-07"),(10,"Eve","Davis","eve.davis@example.com","555-0010","2024-01-12")]
contacts_df=spark.createDataFrame(contacts_data,["contact_id","first_name","last_name","email","phone","created_at"])
display(contacts_df)

window_spec1 = Window.orderBy("first_name","last_name","email")
window_spec2 = Window.partitionBy("first_name","last_name","email")

ranked_df = (
contacts_df
    .withColumn("dup_group_id", dense_rank().over(window_spec1))
    .withColumn("latest_date", max("created_at").over(window_spec2))
    .withColumn("is_primary", when(col("created_at") == col("latest_date"), 1).otherwise(0))
    .select("contact_id", "first_name", "last_name", "email", "dup_group_id", "is_primary")
    .orderBy("first_name", "last_name", "email", "contact_id")
)

display(ranked_df)



contact_id,first_name,last_name,email,phone,created_at
1,John,Doe,john.doe@example.com,555-0001,2024-01-01
2,John,Doe,john.doe@example.com,555-0002,2024-01-05
3,Jane,Smith,jane.smith@example.com,555-0003,2024-01-02
4,Jane,Smith,jane.smith@example.com,555-0004,2024-01-10
5,Bob,Johnson,bob.johnson@example.com,555-0005,2024-01-03
6,Alice,Williams,alice.williams@example.com,555-0006,2024-01-04
7,Alice,Williams,alice.williams@example.com,555-0007,2024-01-08
8,Charlie,Brown,charlie.brown@example.com,555-0008,2024-01-06
9,Eve,Davis,eve.davis@example.com,555-0009,2024-01-07
10,Eve,Davis,eve.davis@example.com,555-0010,2024-01-12


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


contact_id,first_name,last_name,email,dup_group_id,is_primary
6,Alice,Williams,alice.williams@example.com,1,0
7,Alice,Williams,alice.williams@example.com,1,1
5,Bob,Johnson,bob.johnson@example.com,2,1
8,Charlie,Brown,charlie.brown@example.com,3,1
9,Eve,Davis,eve.davis@example.com,4,0
10,Eve,Davis,eve.davis@example.com,4,1
3,Jane,Smith,jane.smith@example.com,5,0
4,Jane,Smith,jane.smith@example.com,5,1
1,John,Doe,john.doe@example.com,6,0
2,John,Doe,john.doe@example.com,6,1


%md
## Que19: Sampling and Stratified Selection

**Difficulty:** Medium

### Problem

A research team needs a reproducible sample from each age-group and gender segment.

For every `(age_group, gender)` combination, return the two people with the smallest `person_id` values. Assign a `stratum_rank` based on the ascending order of `person_id` within each stratum, starting from 1.

If a stratum contains fewer than two people, return all available records.

**Schema columns:** `population.person_id`, `population.age_group`, `population.gender`, `population.region`, `population.income`

**Output columns:** `person_id`, `age_group`, `gender`, `region`, `income`, `stratum_rank`

Order the result by `age_group` in ascending order, then `gender` in ascending order, and finally `person_id` in ascending order.

### Examples

#### Example 1

**Input:**

**population:**

| person_id | age_group | gender | region | income |
|----------:|-----------|--------|--------|--------|
| 1 | 18-25 | M | North | Low |
| 2 | 18-25 | M | North | Medium |
| 3 | 18-25 | M | North | High |
| 13 | 18-25 | F | North | Low |
| 14 | 18-25 | F | North | Medium |
| 15 | 18-25 | F | North | High |

**Output:**

| person_id | age_group | gender | region | income | stratum_rank |
|----------:|-----------|--------|--------|--------|-------------:|
| 13 | 18-25 | F | North | Low | 1 |
| 14 | 18-25 | F | North | Medium | 2 |
| 1 | 18-25 | M | North | Low | 1 |
| 2 | 18-25 | M | North | Medium | 2 |

**Explanation:** The rows are partitioned by `(age_group, gender)`. Within each group, records are ordered by `person_id`, and only the first two are selected. The selected rows receive `stratum_rank` values of 1 and 2.

### Constraints

- Partition records by `(age_group, gender)`.
- Rank records by `person_id` in ascending order within each partition.
- Return at most two records from each partition.
- Assign `stratum_rank` starting from 1 for each partition.
- Order the final result by `age_group`, `gender`, and `person_id` in ascending order.

In [0]:
population_data=[(1,"18-25","M","North","Low"),(2,"18-25","M","North","Medium"),(3,"18-25","M","North","High"),(13,"18-25","F","North","Low"),(14,"18-25","F","North","Medium"),(15,"18-25","F","North","High")]
population_df=spark.createDataFrame(population_data,["person_id","age_group","gender","region","income"])
display(population_df)

window_spec = Window.partitionBy("age_group", "gender").orderBy("person_id")

output_df = (
population_df
    .withColumn("stratum_rank", row_number().over(window_spec))
    .filter(col("stratum_rank") <= 2)
    .orderBy("age_group", "gender", "person_id")
)

display(output_df)



person_id,age_group,gender,region,income
1,18-25,M,North,Low
2,18-25,M,North,Medium
3,18-25,M,North,High
13,18-25,F,North,Low
14,18-25,F,North,Medium
15,18-25,F,North,High


person_id,age_group,gender,region,income,stratum_rank
13,18-25,F,North,Low,1
14,18-25,F,North,Medium,2
1,18-25,M,North,Low,1
2,18-25,M,North,Medium,2


%md
## Que20: Consecutive Numbers

**Difficulty:** Medium

### Problem

A system stores log entries in the order they arrive using an auto-incrementing `id`.

Find every `num` value that appears in at least three consecutive rows when the records are ordered by `id`. A sequence is considered consecutive only if the rows are adjacent by `id`. If the same value appears again later after being interrupted by another value, it starts a new sequence.

Return each qualifying number only once.

**Schema columns:** `logs.id`, `logs.num`

**Output columns:** `consecutive_num`

Order the result by `consecutive_num` in ascending order.

### Examples

#### Example 1

**Input:**

**logs:**

| id | num |
|---:|----:|
| 1 | 1 |
| 2 | 1 |
| 3 | 1 |
| 4 | 2 |
| 5 | 2 |
| 6 | 3 |
| 7 | 3 |
| 8 | 3 |

**Output:**

| consecutive_num |
|----------------:|
| 1 |
| 3 |

**Explanation:** The value `1` appears in three consecutive rows (ids 1, 2, and 3), so it qualifies. The value `3` also appears in three consecutive rows (ids 6, 7, and 8). The value `2` appears only twice consecutively, so it is not included.

### Constraints

- Consider rows in ascending order of `id`.
- A value qualifies only if it appears in three or more consecutive rows.
- Consecutive means adjacent rows by `id`.
- Return each qualifying value only once.
- Order the result by `consecutive_num` in ascending order.

In [0]:
logs_data = [ (1, 1), (2, 1), (3, 1),  (4, 2), (5, 2),  (6, 1), (7, 1),  (8, 3), (9, 3), (10, 3), (11, 3), (12, 4), (13, 5), (14, 5), (15, 5), (16, 6), (17, 6), (18, 5), (19, 5), (20, 5), (21, 5), (22, 7), (23, 8), (24, 8), (25, 9), (26, 10), (27, 10), (28, 10), (29, 11), (30, 12), (31, 11), (32, 11), (33, 11), (34, 13), (35, 13), (36, 13), (37, 13), (38, 13), (39, 14), (40, 14), (41, 15), (42, 14), (43, 14)]
logs_df = spark.createDataFrame(logs_data, ["id", "num"])
# display(logs_df)

window_spec = Window.orderBy("id")

prev_values_df = (
logs_df
    .withColumn("prev_value", lag("num").over(window_spec))
    .withColumn("is_new", when((col("num") != col("prev_value")) | (col("prev_value").isNull()), 1).otherwise(0))
    .withColumn("group_id", sum("is_new").over(window_spec))
)

grouped_df = (
    prev_values_df.groupBy("group_id").agg(
        count("num").alias("total_num")
    )
)

qualified_df = grouped_df.filter(col("total_num") >= 3)

joined_df = qualified_df.alias("q").join(prev_values_df.alias("p"), on=col("q.group_id") == col("p.group_id")).select("p.num").distinct()


display(joined_df)


num
1
3
5
10
11
13


%md
## Que21: Rank Scores

**Difficulty:** Medium

### Problem

A competition stores individual scores and needs a leaderboard of distinct score values.

Assign rank `1` to the highest distinct score and assign consecutive ranks to the remaining distinct scores in descending order without gaps. Duplicate scores should receive the same rank and appear only once in the output.

**Schema columns:** `scores.id`, `scores.score`

**Output columns:** `score`, `dense_rank`

Order the result by `score` in descending order.

### Examples

#### Example 1

**Input:**

**scores:**

| id | score |
|---:|------:|
| 1 | 3.50 |
| 2 | 3.65 |
| 3 | 4.00 |
| 4 | 3.85 |
| 5 | 4.00 |
| 7 | 2.50 |

**Output:**

| score | dense_rank |
|------:|-----------:|
| 4.00 | 1 |
| 3.85 | 2 |
| 3.65 | 3 |
| 3.50 | 4 |
| 2.50 | 5 |

**Explanation:** Duplicate scores are considered only once when assigning ranks. The highest distinct score receives rank 1, and each lower distinct score receives the next consecutive rank without gaps.

### Constraints

- Return one row for each distinct score.
- Equal scores must share the same rank.
- Ranks should be consecutive without gaps.
- Order the result by `score` in descending order.

In [0]:
scores_data=[(1,3.50),(2,3.65),(3,4.00),(4,3.85),(5,4.00),(7,2.50)]
scores_df=spark.createDataFrame(scores_data,["id","score"])
display(scores_df)

window_spec = Window.orderBy(col("score").desc())

output_df = (
scores_df.
    withColumn("dense_rank", dense_rank().over(window_spec))
    .select("score", "dense_rank")
    .distinct()
)

display(output_df)



id,score
1,3.5
2,3.65
3,4.0
4,3.85
5,4.0
7,2.5


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


score,dense_rank
4.0,1
3.85,2
3.65,3
3.5,4
2.5,5


%md
## Que22: Retrieve Alternate Records

**Difficulty:** Medium

### Problem

An analyst wants to retrieve every alternate record from a table.

Order all records by `id` in ascending order and assign a sequential position starting from `1`. Return only the records whose assigned position is odd (1, 3, 5, ...), effectively keeping the first record, skipping the second, keeping the third, and so on.

**Schema columns:** `records.id`, `records.value`, `records.category`

**Output columns:** `row_num`, `id`, `value`, `category`

Order the result by `row_num` in ascending order.

### Examples

#### Example 1

**Input:**

**records:**

| id | value | category |
|---:|------:|----------|
| 1 | 820 | B |
| 2 | 860 | C |
| 3 | 12 | D |
| 4 | 329 | A |
| 5 | 781 | B |
| 6 | 754 | C |

**Output:**

| row_num | id | value | category |
|--------:|---:|------:|----------|
| 1 | 1 | 820 | B |
| 3 | 3 | 12 | D |
| 5 | 5 | 781 | B |

**Explanation:** After ordering the records by `id`, each record is assigned a sequential position starting from 1. Only the records at odd positions (1, 3, and 5) are returned.

### Constraints

- Order records by `id` in ascending order before assigning row numbers.
- Assign `row_num` starting from 1.
- Return only records with odd `row_num` values.
- Order the final result by `row_num` in ascending order.

In [0]:
records_data=[(1,820,"B"),(2,860,"C"),(3,12,"D"),(4,329,"A"),(5,781,"B"),(6,754,"C")]
records_df=spark.createDataFrame(records_data,["id","value","category"])
display(records_df)


window_spec = Window.orderBy(col("id"))

output_df = (
records_df.
    withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") % 2 != 0)
)

display(output_df)



id,value,category
1,820,B
2,860,C
3,12,D
4,329,A
5,781,B
6,754,C


id,value,category,row_num
1,820,B,1
3,12,D,3
5,781,B,5


%md
## Que23: Consecutive Day User Activity

**Difficulty:** Hard

### Problem

A product records the calendar dates on which users are active.

Find all users who were active for at least **three consecutive calendar days**. If a user has multiple activity records on the same date, count that date only once. Return each qualifying user only once.

**Schema columns:** `ca_consecutive_act.date`, `ca_consecutive_act.account_id`, `ca_consecutive_act.user_id`

**Output columns:** `user_id`

Order the result by `user_id` in ascending order.

### Examples

#### Example 1

**Input:**

**ca_consecutive_act:**

| date | account_id | user_id |
|------------|------------|---------|
| 2021-01-10 | A2 | U4 |
| 2021-01-11 | A2 | U4 |
| 2021-01-12 | A2 | U4 |
| 2021-01-01 | A1 | U1 |
| 2021-01-02 | A1 | U1 |
| 2021-01-06 | A1 | U3 |

**Output:**

| user_id |
|---------|
| U4 |

**Explanation:** User **U4** was active on three consecutive calendar days (2021-01-10, 2021-01-11, and 2021-01-12), so the user qualifies. User **U1** has activity on only two consecutive days, while **U3** has activity on a single day.

### Constraints

- Consider duplicate activity records on the same date for a user only once.
- Consecutive days must be exactly one calendar day apart.
- Return users having at least three consecutive active days.
- Return each qualifying user only once.
- Order the result by `user_id` in ascending order.

In [0]:
ca_consecutive_act_df = spark.createDataFrame([("2021-01-01","A1","U1"),("2021-01-02","A1","U1"),("2021-01-03","A1","U1"),("2021-01-03","A1","U1"),("2021-01-05","A1","U1"),("2021-01-06","A1","U1"),("2021-01-10","A1","U1"),("2021-01-11","A1","U1"),("2021-01-12","A1","U1"),("2021-01-13","A1","U1"),("2021-02-01","A2","U2"),("2021-02-03","A2","U2"),("2021-02-04","A2","U2"),("2021-03-01","A3","U3"),("2021-03-02","A3","U3"),("2021-03-04","A3","U3"),("2021-04-01","A4","U4"),("2021-04-02","A4","U4"),("2021-04-03","A4","U4"),("2021-04-04","A4","U4"),("2021-05-01","A5","U5"),("2021-05-01","A5","U5"),("2021-05-02","A5","U5"),("2021-05-03","A5","U5"),("2021-06-01","A6","U6"),("2021-06-02","A6","U6"),("2021-06-04","A6","U6"),("2021-06-05","A6","U6"),("2021-06-06","A6","U6")],["date","account_id","user_id"])


ca_consecutive_act_df = ca_consecutive_act_df.withColumn("date", to_date(col("date"))).select("date", "user_id").distinct()

window_spec = Window.partitionBy("user_id").orderBy("date")

ca_consecutive_act_df = (
ca_consecutive_act_df
    .withColumn("prev_date", lag("date").over(window_spec))
    .withColumn("is_consecutive", 
        when((col("prev_date").isNull()) | (col("date") - 1 != col("prev_date")), 1)
        .otherwise(0)
    )
    .withColumn("group_id", sum("is_consecutive").over(window_spec))
)


grouped_df = (
ca_consecutive_act_df.groupBy("user_id", "group_id").agg(
    count("user_id").alias("total_days")
)
)


output_df = grouped_df.filter(col("total_days") >= 3).select("user_id").distinct().orderBy("user_id")


display(output_df)

user_id
U1
U4
U5
U6


In [0]:
df = (
    ca_consecutive_act_df
    .withColumn("date", to_date("date"))
    .select("user_id", "date")
    .distinct()
)

w = Window.partitionBy("user_id").orderBy("date")

result = (
    df.withColumn("rn", row_number().over(w))
      .withColumn("grp", date_sub(col("date"), col("rn")))
      
)

result.show()

+-------+----------+---+----------+
|user_id|      date| rn|       grp|
+-------+----------+---+----------+
|     U1|2021-01-01|  1|2020-12-31|
|     U1|2021-01-02|  2|2020-12-31|
|     U1|2021-01-03|  3|2020-12-31|
|     U1|2021-01-05|  4|2021-01-01|
|     U1|2021-01-06|  5|2021-01-01|
|     U1|2021-01-10|  6|2021-01-04|
|     U1|2021-01-11|  7|2021-01-04|
|     U1|2021-01-12|  8|2021-01-04|
|     U1|2021-01-13|  9|2021-01-04|
|     U2|2021-02-01|  1|2021-01-31|
|     U2|2021-02-03|  2|2021-02-01|
|     U2|2021-02-04|  3|2021-02-01|
|     U3|2021-03-01|  1|2021-02-28|
|     U3|2021-03-02|  2|2021-02-28|
|     U3|2021-03-04|  3|2021-03-01|
|     U4|2021-04-01|  1|2021-03-31|
|     U4|2021-04-02|  2|2021-03-31|
|     U4|2021-04-03|  3|2021-03-31|
|     U4|2021-04-04|  4|2021-03-31|
|     U5|2021-05-01|  1|2021-04-30|
+-------+----------+---+----------+
only showing top 20 rows


%md
## Que24: Manufacturing Defect Rate Analysis

**Difficulty:** Hard

### Problem

A manufacturing company wants to rank products based on their revenue within each product category.

For every product that has a corresponding sales record, return its category, product name, rounded revenue, and rank within its category. Rank products by revenue in descending order for each category. Products with the same revenue should receive the same rank, and the next rank should be skipped accordingly. Exclude products that do not have any sales record.

**Schema columns:**

- **manufacture_product:** `product_id`, `category`, `product_name`
- **manufacture_sales:** `sale_id`, `product_id`, `quantity`, `revenue`

**Output columns:** `category`, `product_name`, `rank`, `revenue`

Order the result by `category` in ascending order, then by `rank` in ascending order.

### Examples

#### Example 1

**Input:**

**manufacture_product:**

| product_id | category | product_name |
|-----------:|----------|--------------|
| 1 | A | Product1 |
| 2 | A | Product2 |
| 3 | A | Product3 |
| 4 | B | Product4 |
| 5 | B | Product5 |
| 6 | B | Product6 |

**manufacture_sales:**

| sale_id | product_id | quantity | revenue |
|--------:|-----------:|---------:|--------:|
| 1 | 1 | 12 | 180 |
| 2 | 2 | 8 | 120 |
| 3 | 3 | 10 | 120 |
| 4 | 4 | 7 | 69.6 |
| 6 | 6 | 5 | 50 |

**Output:**

| category | product_name | rank | revenue |
|----------|--------------|-----:|--------:|
| A | Product1 | 1 | 180 |
| A | Product2 | 2 | 120 |
| A | Product3 | 2 | 120 |
| B | Product4 | 1 | 70 |
| B | Product6 | 2 | 50 |

**Explanation:** Products are ranked within each category based on revenue in descending order. Equal revenues receive the same rank. Revenue is rounded to the nearest whole number before displaying. Products without a matching sales record are excluded.

### Constraints

- Include only products that have a matching sales record.
- Round revenue to the nearest whole number.
- Rank products within each category by revenue in descending order.
- Products with the same revenue should share the same rank, and the next rank should be skipped.
- Order the final result by `category` and `rank`.

In [0]:
manufacture_product_data=[(1,"A","Product1"),(2,"A","Product2"),(3,"A","Product3"),(4,"B","Product4"),(5,"B","Product5"),(6,"B","Product6")]
manufacture_product_df=spark.createDataFrame(manufacture_product_data,["product_id","category","product_name"])
display(manufacture_product_df)

manufacture_sales_data=[(1,1,12,180.0),(2,2,8,120.0),(3,3,10,120.0),(4,4,7,69.6),(6,6,5,50.0)]
manufacture_sales_df=spark.createDataFrame(manufacture_sales_data,["sale_id","product_id","quantity","revenue"])
display(manufacture_sales_df)

joined_df = (
manufacture_product_df.alias("p")
    .join(manufacture_sales_df.alias("s"), on=col("s.product_id") == col("p.product_id"))
)

window_spec = Window.partitionBy("p.category").orderBy(col("s.revenue").desc())

ranked_df = (
    joined_df.withColumn("rank", rank().over(window_spec))
)

output_df = ranked_df.select("p.category", "p.product_name", "rank", "s.revenue").orderBy("category", "rank")

display(output_df)



product_id,category,product_name
1,A,Product1
2,A,Product2
3,A,Product3
4,B,Product4
5,B,Product5
6,B,Product6


sale_id,product_id,quantity,revenue
1,1,12,180.0
2,2,8,120.0
3,3,10,120.0
4,4,7,69.6
6,6,5,50.0


category,product_name,rank,revenue
A,Product1,1,180.0
A,Product2,2,120.0
A,Product3,2,120.0
B,Product4,1,69.6
B,Product6,2,50.0


%md
## Que25: Country GDP Growth Rate

**Difficulty:** Hard

### Problem

GDP data for countries is received from two separate datasets.

Combine both datasets using **UNION ALL** without removing duplicates. For each country, compare each year's GDP with the GDP from the previous available year for the same country. Calculate the year-over-year GDP growth rate using the formula:


`frac{\text{Current GDP} - \text{Previous GDP}}{\text{Previous GDP}} \times 100`


Round the result to **2 decimal places**. The earliest year for each country has no previous GDP, so its growth rate should be `NULL`.

**Schema columns:**

- **gdp_df1:** `country`, `year`, `gdp`
- **gdp_df2:** `country`, `year`, `gdp`

**Output columns:** `Country`, `Year`, `GDP_growth_rate`

Order the result by `Country` in ascending order, then by `Year` in ascending order.

### Examples

#### Example 1

**Input:**

**gdp_df1:**

| country | year | gdp |
|---------|-----:|--------:|
| USA | 2018 | 20544.34 |
| USA | 2019 | 21427.70 |
| China | 2018 | 13894.04 |

**gdp_df2:**

| country | year | gdp |
|---------|-----:|--------:|
| China | 2019 | 14402.72 |
| India | 2018 | 2713.61 |
| India | 2019 | 2868.93 |

**Output:**

| Country | Year | GDP_growth_rate |
|---------|-----:|----------------:|
| China | 2018 | NULL |
| China | 2019 | 3.66 |
| India | 2018 | NULL |
| India | 2019 | 5.72 |
| USA | 2018 | NULL |
| USA | 2019 | 4.30 |

**Explanation:** Combine both datasets using `UNION ALL`. Within each country, compare every year's GDP with the previous year's GDP. The earliest year for each country has no previous value, so its growth rate is `NULL`.

### Constraints

- Combine `gdp_df1` and `gdp_df2` using `UNION ALL`.
- Do not remove duplicate records.
- Calculate growth using the previous available year for the same country.
- Round the growth rate to 2 decimal places.
- The first year for each country should return `NULL`.
- Order the result by `Country` and `Year`.

In [0]:
gdp_df1_data=[("USA",2018,20544.34),("USA",2019,21427.70),("China",2018,13894.04)]

gdp_df1_df=spark.createDataFrame(gdp_df1_data,["country","year","gdp"])

display(gdp_df1_df)

gdp_df2_data=[("China",2019,14402.72),("India",2018,2713.61),("India",2019,2868.93)]

gdp_df2_df=spark.createDataFrame(gdp_df2_data,["country","year","gdp"])

display(gdp_df2_df)

# UNION ALL (keeps duplicates)
gdp_df = gdp_df1_df.unionByName(gdp_df2_df)

# Window specification
window_spec = Window.partitionBy("country").orderBy("year")

# Calculate previous GDP and growth rate
output_df = (
    gdp_df
    .withColumn("prev_gdp", lag("gdp").over(window_spec))
    .withColumn(
        "GDP_growth_rate",
        when(
            col("prev_gdp").isNull(),
            None
        ).otherwise(
            round(((col("gdp") - col("prev_gdp")) / col("prev_gdp")) * 100, 2)
        )
    )
    .select(
        col("country").alias("Country"),
        col("year").alias("Year"),
        "GDP_growth_rate"
    )
    .orderBy("Country", "Year")
)

display(output_df)

country,year,gdp
USA,2018,20544.34
USA,2019,21427.7
China,2018,13894.04


country,year,gdp
China,2019,14402.72
India,2018,2713.61
India,2019,2868.93


Country,Year,GDP_growth_rate
China,2018,null
China,2019,3.66
India,2018,null
India,2019,5.72
USA,2018,null
USA,2019,4.3


%md
## Que26: Server Utilization Time

**Difficulty:** Hard

### Problem

A server log records alternating **start** and **stop** events for each server.

For every server, pair each `start` event with the immediately following `stop` event and calculate the duration of each session. Sum the durations of all sessions across every server, convert the total uptime into days, and discard any fractional day. Return the final value as `total_uptime_days`.

**Schema columns:** `server_utilization.server_id`, `server_utilization.status_time`, `server_utilization.session_status`

**Output columns:** `total_uptime_days`

### Examples

#### Example 1

**Input:**

**server_utilization:**

| server_id | status_time | session_status |
|----------:|---------------------|----------------|
| 1 | 2024-01-01T08:00:00 | start |
| 1 | 2024-01-02T08:00:00 | stop |
| 1 | 2024-01-03T09:00:00 | start |
| 1 | 2024-01-03T21:00:00 | stop |
| 2 | 2024-01-01T00:00:00 | start |
| 2 | 2024-01-02T12:00:00 | stop |
| 2 | 2024-01-04T10:00:00 | start |
| 2 | 2024-01-04T22:00:00 | stop |

**Output:**

| total_uptime_days |
|------------------:|
| 3 |

**Explanation:** Pair each start event with the next stop event for the same server and calculate the session durations. The combined uptime is 84 hours, which equals 3.5 days. After discarding the fractional part, the final result is 3 days.

### Constraints

- Pair every `start` event with the immediately following `stop` event for the same server.
- Sum the duration of all valid sessions across all servers.
- Convert the total uptime from hours to days.
- Discard any fractional day.
- Return the result as `total_uptime_days`.

In [0]:
server_utilization_data=[(1,"2024-01-01T08:00:00","start"),(1,"2024-01-02T08:00:00","stop"),(1,"2024-01-03T09:00:00","start"),(1,"2024-01-03T21:00:00","stop"),(2,"2024-01-01T00:00:00","start"),(2,"2024-01-02T12:00:00","stop"),(2,"2024-01-04T10:00:00","start"),(2,"2024-01-04T22:00:00","stop")]

server_utilization_df=spark.createDataFrame(server_utilization_data,["server_id","status_time","session_status"])

server_utilization_df = server_utilization_df.withColumn("status_time",to_timestamp("status_time"))

window_spec = Window.partitionBy("server_id").orderBy("status_time")

output_df = (
    server_utilization_df
    .withColumn("next_time", lead("status_time").over(window_spec))
    .filter(col("session_status") == "start")
    .withColumn(
        "time_diff",
        (unix_timestamp("next_time") - unix_timestamp("status_time"))
    )
    .agg(
        floor(sum("time_diff") / (60 * 60 * 24)).alias("total_uptime_days")
    )
)

display(output_df)

total_uptime_days
3


%md
## Que27: Binge Session Detection

**Difficulty:** Hard

### Problem

Identify binge-watching sessions for each user and series.

Episodes belong to the same binge session if the next episode starts within **30 minutes** after the previous episode ends. Group consecutive episodes into sessions and return only those sessions that contain **at least three episodes**.

**Schema columns:** `viewing_events.event_id`, `viewing_events.user_id`, `viewing_events.series_id`, `viewing_events.episode_num`, `viewing_events.start_time`, `viewing_events.end_time`

**Output columns:** `user_id`, `series_id`, `episode_count`, `session_start`, `session_end`

Order the result by `user_id` in ascending order, then by `session_start` in ascending order.

### Examples

#### Example 1

**Input:**

**viewing_events:**

| user_id | end_time | event_id | series_id | start_time | episode_num |
|--------:|---------------------|---------:|----------:|---------------------|------------:|
| 1 | 2024-01-01T09:00:00 | 1 | 1 | 2024-01-01T08:00:00 | 1 |
| 1 | 2024-01-01T10:00:00 | 2 | 1 | 2024-01-01T09:15:00 | 2 |
| 1 | 2024-01-01T11:00:00 | 3 | 1 | 2024-01-01T10:10:00 | 3 |
| 1 | 2024-01-01T15:00:00 | 4 | 1 | 2024-01-01T14:00:00 | 5 |
| 1 | 2024-01-01T16:00:00 | 5 | 1 | 2024-01-01T15:10:00 | 6 |
| 1 | 2024-01-01T17:00:00 | 6 | 1 | 2024-01-01T16:15:00 | 7 |
| 2 | 2024-01-02T11:00:00 | 7 | 2 | 2024-01-02T10:00:00 | 1 |
| 2 | 2024-01-02T12:00:00 | 8 | 2 | 2024-01-02T11:15:00 | 2 |

**Output:**

| user_id | series_id | episode_count | session_start | session_end |
|--------:|----------:|--------------:|---------------------|---------------------|
| 1 | 1 | 3 | 2024-01-01 08:00:00 | 2024-01-01 11:00:00 |
| 1 | 1 | 3 | 2024-01-01 14:00:00 | 2024-01-01 17:00:00 |

**Explanation:** Consecutive episodes separated by less than 30 minutes are grouped into the same session. Only sessions containing at least three episodes are returned.

### Constraints

- Partition sessions by both `user_id` and `series_id`.
- Consecutive episodes belong to the same session only if the next episode starts within 30 minutes of the previous episode ending.
- Return only sessions with at least three episodes.
- Return `user_id`, `series_id`, `episode_count`, `session_start`, and `session_end`.
- Order the result by `user_id` and `session_start`.

In [0]:
viewing_events_data=[(1,"2024-01-01T09:00:00",1,1,"2024-01-01T08:00:00",1),(1,"2024-01-01T10:00:00",2,1,"2024-01-01T09:15:00",2),(1,"2024-01-01T11:00:00",3,1,"2024-01-01T10:10:00",3),(1,"2024-01-01T15:00:00",4,1,"2024-01-01T14:00:00",5),(1,"2024-01-01T16:00:00",5,1,"2024-01-01T15:10:00",6),(1,"2024-01-01T17:00:00",6,1,"2024-01-01T16:15:00",7),(2,"2024-01-02T11:00:00",7,2,"2024-01-02T10:00:00",1),(2,"2024-01-02T12:00:00",8,2,"2024-01-02T11:15:00",2)]

viewing_events_df=spark.createDataFrame(viewing_events_data,["user_id","end_time","event_id","series_id","start_time","episode_num"])

viewing_events_df = (
viewing_events_df
    .withColumn("start_time", to_timestamp("start_time"))
    .withColumn("end_time", to_timestamp("end_time"))
    .select("user_id","event_id","series_id","episode_num", "start_time", "end_time")
)



window = Window.partitionBy("user_id", "series_id").orderBy("start_time")

prev_end_time_df = (
viewing_events_df
    .withColumn("prev_end_time", lag("end_time").over(window))
    .withColumn("time_diff", (unix_timestamp("start_time") - unix_timestamp("prev_end_time")) / 60)
    .withColumn("is_new_start", 
        when(col("time_diff").isNull() | (col("time_diff") > 30), 1).otherwise(0)
    )
    .withColumn("group_id", sum("is_new_start").over(window))
)

output_df = (
prev_end_time_df.groupBy("user_id", "series_id", "group_id").agg(
    count("episode_num").alias("episode_count"),
    min("start_time").alias("session_start"),
    max("end_time").alias("session_end")
)
.filter(col("episode_count") >= 3)
.drop("group_id")
.orderBy("user_id", "session_start")

)


display(output_df)

user_id,series_id,total_episode,session_start,session_end
1,1,3,2024-01-01T08:00:00.000Z,2024-01-01T11:00:00.000Z
1,1,3,2024-01-01T14:00:00.000Z,2024-01-01T17:00:00.000Z


%md
## Que28: Window Function - Row Number and Ranking

**Difficulty:** Hard

### Problem

Create a salary leaderboard for employees within each department.

For every employee, generate three ranking values based on salary in descending order within the same department:
- `row_num`: Assign a unique sequential number, breaking salary ties using `emp_id`.
- `rank`: Employees with the same salary receive the same rank, and the next rank is skipped.
- `dense_rank`: Employees with the same salary receive the same rank, but the next rank is not skipped.

**Schema columns:** `employees.emp_id`, `employees.name`, `employees.department`, `employees.salary`

**Output columns:** `emp_id`, `name`, `department`, `salary`, `row_num`, `rank`, `dense_rank`

Order the result by `department` in ascending order, then by `salary` in descending order, and finally by `emp_id` in ascending order.

### Examples

#### Example 1

**Input:**

**employees:**

| name | emp_id | salary | department |
|------|-------:|-------:|------------|
| Employee_1 | 1 | 50000 | Sales |
| Employee_2 | 2 | 75000 | IT |
| Employee_3 | 3 | 60000 | HR |
| Employee_4 | 4 | 55000 | Sales |
| Employee_5 | 5 | 75000 | IT |
| Employee_6 | 6 | 62000 | HR |
| Employee_7 | 7 | 52000 | Sales |
| Employee_8 | 8 | 80000 | IT |

**Output:**

| emp_id | name | department | salary | row_num | rank | dense_rank |
|-------:|------|------------|-------:|--------:|----:|-----------:|
| 6 | Employee_6 | HR | 62000 | 1 | 1 | 1 |
| 3 | Employee_3 | HR | 60000 | 2 | 2 | 2 |
| 8 | Employee_8 | IT | 80000 | 1 | 1 | 1 |
| 2 | Employee_2 | IT | 75000 | 2 | 2 | 2 |
| 5 | Employee_5 | IT | 75000 | 3 | 2 | 2 |
| 4 | Employee_4 | Sales | 55000 | 1 | 1 | 1 |
| 7 | Employee_7 | Sales | 52000 | 2 | 2 | 2 |
| 1 | Employee_1 | Sales | 50000 | 3 | 3 | 3 |

**Explanation:** Employees are ranked independently within each department. `row_num` always produces unique values, while `rank` and `dense_rank` handle salary ties differently. In the IT department, employees 2 and 5 have the same salary, so they share the same `rank` and `dense_rank`, but receive different `row_num` values.

### Constraints

- Perform ranking separately for each department.
- Sort employees by salary in descending order before ranking.
- Break salary ties using `emp_id` only for `row_num`.
- `rank` should skip positions after ties, while `dense_rank` should not.
- Order the final result by `department`, `salary` (descending), and `emp_id`.

In [0]:
employees_data=[("Employee_1",1,50000,"Sales"),("Employee_2",2,75000,"IT"),("Employee_3",3,60000,"HR"),("Employee_4",4,55000,"Sales"),("Employee_5",5,75000,"IT"),("Employee_6",6,62000,"HR"),("Employee_7",7,52000,"Sales"),("Employee_8",8,80000,"IT")]

employees_df=spark.createDataFrame(employees_data,["name","emp_id","salary","department"])

# Window for row_number (break ties using emp_id)
row_window = Window.partitionBy("department") \
                   .orderBy(col("salary").desc(), col("emp_id"))

# Window for rank and dense_rank (ties only on salary)
rank_window = Window.partitionBy("department") \
                    .orderBy(col("salary").desc())

output_df = (
    employees_df
    .withColumn("row_num", row_number().over(row_window))
    .withColumn("rank", rank().over(rank_window))
    .withColumn("dense_rank", dense_rank().over(rank_window))
    .select(
        "emp_id",
        "name",
        "department",
        "salary",
        "row_num",
        "rank",
        "dense_rank"
    )
    .orderBy("department", col("salary").desc(), "emp_id")
)

display(output_df)

emp_id,name,department,salary,row_num,rank,dense_rank
6,Employee_6,HR,62000,1,1,1
3,Employee_3,HR,60000,2,2,2
8,Employee_8,IT,80000,1,1,1
2,Employee_2,IT,75000,2,2,2
5,Employee_5,IT,75000,3,2,2
4,Employee_4,Sales,55000,1,1,1
7,Employee_7,Sales,52000,2,2,2
1,Employee_1,Sales,50000,3,3,3


## Que29: Cumulative Rank with Reset (Session Running Total)

**Difficulty:** Hard

### Problem

A product analytics system records user login and purchase events.

For each user, every `login` event starts a new session. The first login creates **session 1**, and each subsequent login increments the session number. Purchase events belong to the most recently started session. If an event occurs before the user's first login, it belongs to **session 0**.

Return every event along with its session number and the cumulative purchase amount within that session. The running total should reset whenever a new login starts.

**Schema columns:** `user_sessions.event_id`, `user_sessions.user_id`, `user_sessions.event_date`, `user_sessions.event_type`, `user_sessions.amount`

**Output columns:** `user_id`, `event_date`, `event_type`, `amount`, `session_id`, `session_running_total`

Order the result by `user_id` in ascending order, then by `event_date` in ascending order, and finally by `event_id` in ascending order.

### Examples

#### Example 1

**Input:**

**user_sessions:**

| event_id | user_id | event_date | event_type | amount |
|---------:|--------:|------------|------------|-------:|
| 1 | 1 | 2024-01-01 | login | 0.00 |
| 2 | 1 | 2024-01-01 | purchase | 50.00 |
| 3 | 1 | 2024-01-01 | purchase | 30.00 |
| 4 | 1 | 2024-01-02 | login | 0.00 |
| 5 | 1 | 2024-01-02 | purchase | 75.00 |

**Output:**

| user_id | event_date | event_type | amount | session_id | session_running_total |
|--------:|------------|------------|-------:|-----------:|----------------------:|
| 1 | 2024-01-01 | login | 0.00 | 1 | 0.00 |
| 1 | 2024-01-01 | purchase | 50.00 | 1 | 50.00 |
| 1 | 2024-01-01 | purchase | 30.00 | 1 | 80.00 |
| 1 | 2024-01-02 | login | 0.00 | 2 | 0.00 |
| 1 | 2024-01-02 | purchase | 75.00 | 2 | 75.00 |

**Explanation:** Each login starts a new session. Purchase amounts accumulate within the current session, while login events reset the running total to zero. Purchases after the second login begin a new cumulative total.

### Constraints

- Process events for each user in ascending order of `event_date` and `event_id`.
- The first login starts session 1, and every subsequent login increments the session number.
- Events before the first login belong to session 0.
- Running totals include only purchase amounts and reset after each login.
- Order the final result by `user_id`, `event_date`, and `event_id`.

In [0]:
user_sessions_data = [(1,  101, "2024-01-01", "purchase", 25.00),(2,  101, "2024-01-01", "purchase", 15.00),(3,  101, "2024-01-01", "login",     0.00),(4,  101, "2024-01-01", "purchase", 100.00),(5,  101, "2024-01-01", "purchase", 50.00),(6,  101, "2024-01-01", "login",     0.00),(7,  101, "2024-01-01", "purchase", 20.00),(8,  101, "2024-01-01", "login",     0.00),(9,  101, "2024-01-01", "login",     0.00),(10, 101, "2024-01-01", "purchase", 30.00),(11, 102, "2024-01-02", "login",     0.00),(12, 102, "2024-01-02", "purchase", 200.00),(13, 102, "2024-01-02", "purchase", 300.00),(14, 102, "2024-01-02", "login",     0.00),(15, 102, "2024-01-02", "purchase", 50.00),(16, 102, "2024-01-02", "purchase", 25.00),(17, 103, "2024-01-03", "purchase", 10.00),(18, 103, "2024-01-03", "purchase", 20.00),(19, 103, "2024-01-03", "login",     0.00),(20, 103, "2024-01-03", "purchase", 40.00),(21, 103, "2024-01-03", "login",     0.00),(22, 103, "2024-01-03", "purchase", 60.00),(23, 103, "2024-01-03", "purchase", 40.00),(24, 104, "2024-01-04", "login",     0.00),(25, 104, "2024-01-04", "login",     0.00),(26, 104, "2024-01-04", "login",     0.00),(27, 104, "2024-01-04", "purchase", 500.00),(28, 105, "2024-01-05", "purchase", 100.00),(29, 105, "2024-01-05", "login",      0.00),(30, 105, "2024-01-05", "purchase", 50.00),(31, 105, "2024-01-05", "login",      0.00),(32, 105, "2024-01-05", "purchase", 25.00)]

user_sessions_df=spark.createDataFrame(user_sessions_data,["event_id","user_id","event_date","event_type","amount"])

session_window = Window.partitionBy("user_id").orderBy("event_date", "event_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

user_sessions_df = (
user_sessions_df
    .withColumn("is_new_session", when(col("event_type") == 'login', 1).otherwise(0))
    .withColumn("session_id", sum("is_new_session").over(session_window))
    .drop("is_new_session")
)

sum_window = Window.partitionBy("user_id", "session_id").orderBy("event_date", "event_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

output_df = (
user_sessions_df
    .withColumn("session_running_total", sum("amount").over(sum_window))
    .orderBy("user_id", "event_date", "event_id")
)

display(output_df)



event_id,user_id,event_date,event_type,amount,session_id,session_running_total
1,101,2024-01-01,purchase,25.0,0,25.0
2,101,2024-01-01,purchase,15.0,0,40.0
3,101,2024-01-01,login,0.0,1,0.0
4,101,2024-01-01,purchase,100.0,1,100.0
5,101,2024-01-01,purchase,50.0,1,150.0
6,101,2024-01-01,login,0.0,2,0.0
7,101,2024-01-01,purchase,20.0,2,20.0
8,101,2024-01-01,login,0.0,3,0.0
9,101,2024-01-01,login,0.0,4,0.0
10,101,2024-01-01,purchase,30.0,4,30.0


%md
## Que30: Nth Highest Salary

**Difficulty:** Hard

### Problem

A compensation analyst wants to determine the company's salary bands by identifying the **3rd highest distinct salary** across all employees.

Employees earning the same salary should be treated as a single distinct salary value. If there are fewer than three distinct salary values in the company, return `NULL`.

**Schema columns:** `employees.emp_id`, `employees.emp_name`, `employees.salary`

**Output columns:** `nth_highest_salary`

### Examples

#### Example 1

**Input:**

**employees:**

| emp_id | emp_name | salary |
|-------:|-----------|-------:|
| 1 | Employee_1 | 90000 |
| 2 | Employee_2 | 90000 |
| 3 | Employee_3 | 70000 |
| 4 | Employee_4 | 50000 |
| 5 | Employee_5 | 30000 |

**Output:**

| nth_highest_salary |
|-------------------:|
| 50000 |

**Explanation:** The distinct salaries in descending order are 90000, 70000, 50000, and 30000. Since duplicate salary values are counted only once, the third highest distinct salary is 50000.

### Constraints

- Consider only distinct salary values.
- Employees with the same salary contribute a single distinct salary.
- Return `NULL` if fewer than three distinct salaries exist.
- Return a single row containing `nth_highest_salary`.

In [0]:
employees_data=[(1,"Employee_1",90000),(2,"Employee_2",90000),(3,"Employee_3",70000),(4,"Employee_4",50000),(5,"Employee_5",30000)]
employees_df=spark.createDataFrame(employees_data,["emp_id","emp_name","salary"])
display(employees_df)

window_spec = Window.orderBy(col("salary").desc())

employees_df.withColumn("rank",dense_rank().over(window_spec)).filter("rank = 3").select(col("salary").alias("nth_highest_salary")).display()


emp_id,emp_name,salary
1,Employee_1,90000
2,Employee_2,90000
3,Employee_3,70000
4,Employee_4,50000
5,Employee_5,30000


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


emp_id,emp_name,salary,rank
1,Employee_1,90000,1
2,Employee_2,90000,1
3,Employee_3,70000,2
4,Employee_4,50000,3
5,Employee_5,30000,4
